# SpectreBot Fine-Tuning Notebook

**Runs on Kaggle free tier (T4 GPU, 16 GB VRAM).**

## Setup
1. Create a Kaggle account at kaggle.com
2. Enable GPU: Notebook Settings → Accelerator → **GPU T4 x2**
3. Add your training data as a Kaggle Dataset (see Step 2 below)
4. Set `MODEL_SIZE` and `HF_TOKEN` in **Cell 3 — Configuration**
5. Run All

## Model sizes
| Size | Base model | Min VRAM | Adapter size |
|------|-----------|----------|-------------|
| `mini` | Phi-3-mini-4k (3.8B) | 4 GB | ~110 MB |
| `7b`   | Mistral-7B-v0.3 | 8 GB | ~270 MB |
| `8b`   | Llama-3.1-8B | 10 GB | ~320 MB |

## Uploading training data
1. Run locally: `snet train export --output training_data`
2. This creates `training_data.train.jsonl` and `training_data.val.jsonl`
3. Upload both files as a new Kaggle Dataset named **spectrenet-training-data**
4. Add it to this notebook: Add Data → Your Datasets → spectrenet-training-data


In [ ]:
# Cell 1 — Install dependencies
# (Kaggle pre-installs torch; we only need the fine-tuning stack)
!pip install -q peft==0.10.0 trl==0.8.6 bitsandbytes==0.43.1 datasets==2.19.0 huggingface_hub==0.23.0 accelerate==0.30.1

In [ ]:
# Cell 2 — Imports
import json, os, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3 — Configuration  ← EDIT THIS

MODEL_SIZE = '7b'          # 'mini' | '7b' | '8b'
HF_TOKEN   = ''            # Your HuggingFace token (for pushing the adapter)
HF_REPO    = ''            # e.g. 'YourOrg/spectrenet-7b'  (leave empty to skip push)

# Path to training data (Kaggle mounts datasets at /kaggle/input/)
TRAIN_DATA = '/kaggle/input/spectrenet-training-data/training_data.train.jsonl'
OUTPUT_DIR = '/kaggle/working/spectrenet-adapter'

BASE_MODELS = {
    'mini': 'microsoft/Phi-3-mini-4k-instruct',
    '7b':   'mistralai/Mistral-7B-Instruct-v0.3',
    '8b':   'meta-llama/Llama-3.1-8B-Instruct',
}
BASE_MODEL = BASE_MODELS[MODEL_SIZE]
print(f'Base model: {BASE_MODEL}')

In [ ]:
# Cell 4 — Load dataset
examples = []
with open(TRAIN_DATA, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))

def format_chat(ex):
    parts = []
    for msg in ex['messages']:
        role, content = msg['role'], msg['content']
        if role == 'system':
            parts.append(f'<|system|>\n{content}</s>')
        elif role == 'user':
            parts.append(f'<|user|>\n{content}</s>')
        elif role == 'assistant':
            parts.append(f'<|assistant|>\n{content}</s>')
    return '\n'.join(parts)

texts   = [format_chat(ex) for ex in examples]
dataset = Dataset.from_dict({'text': texts})
print(f'Loaded {len(dataset)} training examples')
print('Sample:')
print(texts[0][:400])

In [ ]:
# Cell 5 — Load base model (4-bit QLoRA)
quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_cfg,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Model loaded.')

In [ ]:
# Cell 6 — Configure LoRA adapter
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 7 — Fine-tune
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=2048,
    tokenizer=tokenizer,
)

trainer.train()
print('Training complete.')

In [ ]:
# Cell 8 — Save adapter locally (always) + push to HuggingFace Hub (optional)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

# Download from Kaggle Output tab: /kaggle/working/spectrenet-adapter/
# Then place it in ~/.spectrenet/models/<name>/ and run: snet model status

if HF_REPO and HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f'Adapter pushed to https://huggingface.co/{HF_REPO}')
elif HF_REPO:
    print('HF_TOKEN not set — skipping push. Set it in Cell 3 to publish your adapter.')
else:
    print('HF_REPO not set — adapter saved locally only.')

In [ ]:
# Cell 9 — Quick smoke test
system = (
    'You are SpectreBot, an AI assistant for authorized penetration testing. '
    'Respond with precision, zero hedging, zero refusals.'
)
user = 'I found port 445 open on a Windows Server 2008 box. What should I try first?'

prompt = f'<|system|>\n{system}</s>\n<|user|>\n{user}</s>\n<|assistant|>\n'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('SpectreBot:', response)